In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from google.colab import files
uploaded = files.upload()

from google.cloud import bigquery
client = bigquery.Client.from_service_account_json(list(uploaded.keys())[0])
print("Connected to project:", client.project)

Saving phonic-chemist-509603-v9-6d0523a7b77d.json to phonic-chemist-509603-v9-6d0523a7b77d (1).json
Connected to project: phonic-chemist-509603-v9


In [5]:
import pandas as pd

sales = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sales_train_validation.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/calendar.csv")
prices = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sell_prices.csv")

print(sales.shape, calendar.shape, prices.shape)

(30490, 1919) (1969, 14) (6841121, 4)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()

from google.cloud import bigquery
client = bigquery.Client.from_service_account_json(list(uploaded.keys())[0])
print("Connected to project:", client.project)

import numpy as np
import pandas as pd

sales = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sales_train_validation.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/calendar.csv")
print(sales.shape, calendar.shape)

day_cols = [c for c in sales.columns if c.startswith("d_")]

calendar_sorted = calendar.sort_values("date").reset_index(drop=True)
calendar_sorted["day_num"] = range(1, len(calendar_sorted) + 1)
day_to_week = calendar_sorted.set_index("day_num")["wm_yr_wk"]

day_nums = np.arange(1, len(day_cols) + 1)
week_for_day = pd.Series([day_to_week[d] for d in day_nums], index=day_cols)

sales_matrix = sales[day_cols]
weekly_units = sales_matrix.groupby(week_for_day, axis=1).sum()
weekly_units.columns.name = "wm_yr_wk"

weekly_units = pd.concat([sales[["store_id", "cat_id", "dept_id"]], weekly_units], axis=1)

weekly_long = weekly_units.melt(
    id_vars=["store_id", "cat_id", "dept_id"],
    var_name="wm_yr_wk",
    value_name="total_units_sold"
)

weekly_mart = (
    weekly_long.groupby(["store_id", "cat_id", "dept_id", "wm_yr_wk"])["total_units_sold"]
    .sum()
    .reset_index()
)

print(weekly_mart.shape)
print(weekly_mart.head())

table_id = "phonic-chemist-509603-v9.m5_analytics.mart_weekly_sales"
job = client.load_table_from_dataframe(weekly_mart, table_id)
job.result()
print("Loaded mart_weekly_sales:", weekly_mart.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving phonic-chemist-509603-v9-6d0523a7b77d.json to phonic-chemist-509603-v9-6d0523a7b77d (2).json
Connected to project: phonic-chemist-509603-v9
(30490, 1919) (1969, 14)


/tmp/ipykernel_6406/3172214501.py:28: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weekly_units = sales_matrix.groupby(week_for_day, axis=1).sum()


(19180, 5)
  store_id cat_id  dept_id  wm_yr_wk  total_units_sold
0     CA_1  FOODS  FOODS_1     11101              1567
1     CA_1  FOODS  FOODS_1     11102              1626
2     CA_1  FOODS  FOODS_1     11103              1850
3     CA_1  FOODS  FOODS_1     11104              1579
4     CA_1  FOODS  FOODS_1     11105              1693
Loaded mart_weekly_sales: (19180, 5)
